#TASK 1 - LOAd AND INSPECT

In [25]:
import pandas as pd
 #for reading the csv file
data=pd.read_csv('bangalore_tech_salaries.csv')
data
#for basic information
data.info()

data.shape

data.dtypes

data.isnull().sum()

data.duplicated().sum()

data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB


,years_exp,Joining_Year
count,1015.000000,1015.000000
mean,3.429557,2022.698522
std,2.558721,1.308351
min,0.000000,2020.000000
25%,1.000000,2022.000000
50%,3.000000,2023.000000
75%,5.000000,2024.000000
max,12.000000,2024.000000


In [27]:
data.duplicated().sum()

np.int64(15)

#DATASET QUALITY SUMMARY:

~ The dataset contains 1015 rows and 12 columns.

~ Datatypes in the dataset are 'object' and 'int64'.


~ "Previous_CTC" has 200 missing values,"Skills" has 27 missing values and
"Location" has 20 missing values.

~ There are 15 duplicate rows in the dataset.



#TASK 2 - CLEAN

In [43]:
#Rename columns to snake_case.
data=data.copy()
data.columns=data.columns.str.strip().str.lower()
print(data.columns.tolist())

def clean_ctc_column(column_series):
  cleaned_strings = column_series.astype(str).str.replace(',','',regex=False).str.strip()
  numeric_values=pd.to_numeric(cleaned_strings,errors='coerce')
  return numeric_values

data['previous_ctc']=clean_ctc_column(data['previous_ctc'])
data['current_ctc']=clean_ctc_column(data['current_ctc'])
data['role']=data['role'].astype(str).str.strip()
role_mapping={
    "Ds": "Data Scientist",
    "Data Scientist": "Data Scientist",
    "DA": "Data Analyst",
    "Data Analyst": "Data Analyst",
    "MLE": "Machine Learning Engineer",
    "Machine Learning Engineer": "Machine Learning Engineer",
    "sDE": "Software Development Engineer",
    "Software Development Engineer": "Software Development Engineer",
    "DevOps Engineer": "DevOps Engineer",
    "DevOps Engineer": "DevOps Engineer",
}

data["role"] = data["role"].replace(role_mapping)
print(data["company_type"].unique())

company_mapping={
    "mnc":"MNC",
    "MNC": "MNC",
    "unicorn": "Unicorn",
    "UNICORN": "Unicorn",
    "Unicorn": "Unicorn",
    "mid-size": "Mid-size",
    "MID-SIZE": "Mid-size",
    "Mid size": "Mid-size",
    "early-stage": "Early-stage",
    "Early-stage": "Early-stage",
    "EARLY-STAGE": "Early-stage"
}
data["company_type"]=data["company_type"].replace(company_mapping)
print(data["education_tier"].unique())

education_mapping={
    "1": "Tier-1",
    "2": "Tier-2",
    "3": "Tier-3",
    "Tier-1": "Tier-1",
    "Tier-2": "Tier-2",
    "Tier-3": "Tier-3",
    "T1": "Tier-1",
    "T2": "Tier-2",
    "T3": "Tier-3"
}
data["education_tier"]=data["education_tier"].astype(str).str.strip().replace(education_mapping)


data=data.drop_duplicates()

print(data.dtypes)
print(data["role"].value_counts())
print(data["company_type"].value_counts())
print(data["education_tier"].value_counts())



['employee_id', 'role', 'years_exp', 'current_ctc', 'previous_ctc', 'company', 'company_type', 'skills', 'location', 'education_tier', 'joining_year', 'work_mode']
['Unicorn' 'MNC' 'Mid-size' 'Early-stage']
['Tier-1' 'Tier-2' 'Tier 1' 'Tier 2' 'Tier-3' 'Tier 3']
employee_id        object
role               object
years_exp           int64
current_ctc       float64
previous_ctc      float64
company            object
company_type       object
skills             object
location           object
education_tier     object
joining_year        int64
work_mode          object
dtype: object
role
Data Analyst                 50
BA                           41
Data Scientist               39
Business Analyst             35
Site Reliability Engineer    33
Business Systems Analyst     32
Product Lead                 31
ML Engineer                  31
Data Science Engineer        30
PM                           28
SRE                          28
Designer                     28
BI Analyst            

#TASK - 3

In [47]:
#Q 3.1

# Group by role and calculate salary statistics
role_salary = data.groupby("role")["current_ctc"].agg(
    median="median",
    mean="mean",
    min="min",
    max="max"
)

# Sort by median CTC
role_salary = role_salary.sort_values(by="median", ascending=False)

# Display the complete table
print("CTC Distribution by Role")
print(role_salary)

# Highest paying role
highest_role = role_salary.index[0]
highest_median = role_salary.iloc[0]["median"]

print("\nHighest Paying Role:")
print(highest_role)
print("Median CTC:", highest_median)

# Lowest paying role
lowest_role = role_salary.index[-1]
lowest_median = role_salary.iloc[-1]["median"]

print("\nLowest Paying Role:")
print(lowest_role)
print("Median CTC:", lowest_median)

# Check for possible outliers
print("\nRoles Showing Possible Outliers")

for role in role_salary.index:
    median = role_salary.loc[role, "median"]
    mean = role_salary.loc[role, "mean"]

    if abs(mean - median) > (0.20 * median):
        print(role)
        print("Median:", median)
        print("Mean:", mean)
        print()

CTC Distribution by Role
                               median          mean   min        max
role                                                                
PM                         1100028.75  1.820731e+06  25.0  5690000.0
SDE FS                     1090023.15  2.132516e+06  18.3  7090000.0
BI Analyst                  995000.00  1.320840e+06   7.2  3629999.0
Site Reliability Engineer   755026.00  1.019386e+06   9.9  3270000.0
UX Designer                 695015.90  8.016770e+05  13.5  1839999.0
SDE-Frontend                605007.90  7.925072e+05  13.0  1960000.0
Data Scientist                  73.50  1.160016e+06  16.0  4170000.0
Sr PM                           45.90  1.061130e+06  10.8  4730000.0
Product Lead                    41.10  1.208841e+06  12.0  4370000.0
DS                              39.65  6.075186e+05  10.9  2730000.0
BA                              37.30  1.124104e+06   7.6  5270000.0
Data Science Engineer           37.20  7.807327e+05  15.4  2900000.0
ML Engine

#Insight

PM has the highest median CTC, while the lowest-paying role appears at the bottom of the sorted table. Most roles show a very large difference between mean and median CTC, indicating the presence of extreme salary values or inconsistent salary units in the dataset.

In [48]:
# Filter only SDE Backend employees
backend = data[data["role"] == "SDE Backend"].copy()

# Create experience buckets
bins = [-1, 1, 3, 5, 100]
labels = ["0-1", "2-3", "4-5", "6+"]

backend["exp_band"] = pd.cut(
    backend["years_exp"],
    bins=bins,
    labels=labels
)

# Median CTC for each experience bucket
median_ctc = backend.groupby("exp_band")["current_ctc"].median()

print("Median CTC by Experience Band")
print(median_ctc)

# Calculate growth between consecutive experience bands
print("\nCTC Growth Between Experience Bands")

for i in range(1, len(median_ctc)):
    previous = median_ctc.iloc[i-1]
    current = median_ctc.iloc[i]

    growth = ((current - previous) / previous) * 100

    print(
        median_ctc.index[i-1],
        "to",
        median_ctc.index[i],
        ":",
        round(growth, 2),
        "%"
    )

Median CTC by Experience Band
exp_band
0-1         16.20
2-3         23.30
4-5         27.25
6+     3360000.00
Name: current_ctc, dtype: float64

CTC Growth Between Experience Bands
0-1 to 2-3 : 43.83 %
2-3 to 4-5 : 16.95 %
4-5 to 6+ : 12330175.23 %


/tmp/ipykernel_1357/2838862876.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  median_ctc = backend.groupby("exp_band")["current_ctc"].median()


#Insight:

The median CTC generally increases as experience increases. The highest median salary is observed in the ____ experience band, indicating that professionals with more experience tend to earn higher salaries.

In [49]:
# Filter SDE employees
sde = data[
    data["role"].isin([
        "SDE Backend",
        "SDE Frontend",
        "SDE Full-Stack"
    ])
].copy()

# Skills to compare
skills = ["AWS", "ML", "System Design", "Kubernetes"]

result = []

for skill in skills:

    with_skill = sde[
        sde["skills"].str.contains(skill, case=False, na=False)
    ]["current_ctc"].median()

    without_skill = sde[
        ~sde["skills"].str.contains(skill, case=False, na=False)
    ]["current_ctc"].median()

    premium = with_skill - without_skill

    result.append([skill, with_skill, without_skill, premium])

result_df = pd.DataFrame(
    result,
    columns=[
        "Skill",
        "Median CTC (With Skill)",
        "Median CTC (Without Skill)",
        "Premium"
    ]
)

result_df = result_df.sort_values(
    by="Premium",
    ascending=False
)

print(result_df)

print("\nSkill with Highest Premium:")
print(result_df.iloc[0]["Skill"])

           Skill  Median CTC (With Skill)  Median CTC (Without Skill)  Premium
1             ML                    31.80                       26.05     5.75
0            AWS                    27.15                       26.20     0.95
2  System Design                    21.40                       27.30    -5.90
3     Kubernetes                    16.50                       28.15   -11.65

Skill with Highest Premium:
ML


#Insight:

Among the selected skills, ML provides the highest median CTC premium for SDE professionals. Employees possessing this skill generally earn a higher median salary than those who do not have it.

In [50]:
# Select one role (Data Scientist)
role_data = data[data["role"] == "Data Scientist"].copy()

# Median CTC by company type
company_ctc = role_data.groupby("company_type")["current_ctc"].median()

print("Median CTC by Company Type")
print(company_ctc)

# Unicorn and MNC median CTC
unicorn_ctc = company_ctc["Unicorn"]
mnc_ctc = company_ctc["MNC"]

premium = ((unicorn_ctc - mnc_ctc) / mnc_ctc) * 100

print("\nMedian CTC in Unicorn:", unicorn_ctc)
print("Median CTC in MNC:", mnc_ctc)
print("Unicorn Premium over MNC:", round(premium, 2), "%")

Median CTC by Company Type
company_type
Early-stage    1939999.0
MNC                 24.2
Mid-size            55.0
Unicorn        1370000.0
Name: current_ctc, dtype: float64

Median CTC in Unicorn: 1370000.0
Median CTC in MNC: 24.200000000000003
Unicorn Premium over MNC: 5661057.02 %


#Insight:

Insight:

For the Data Scientist role, the median CTC in Unicorn companies is higher than in MNCs. However, the calculated premium is unusually large because the dataset contains inconsistent salary units (lakhs and rupees), making the comparison unreliable until the CTC values are standardized.

In [52]:
# Create experience bands
bins = [-1, 1, 3, 5, 100]
labels = ["0-1", "2-3", "4-5", "6+"]

data["exp_band"] = pd.cut(
    data["years_exp"],
    bins=bins,
    labels=labels
)

# Count members in each group
group_size = data.groupby(
    ["role", "company_type", "exp_band"]
)["current_ctc"].transform("count")

# Use groups with at least 10 members if available,
# otherwise use groups with at least 5 members.
if (group_size >= 10).any():
    filtered = data[group_size >= 10].copy()
else:
    filtered = data[group_size >= 5].copy()

# Calculate group median
filtered["group_median"] = filtered.groupby(
    ["role", "company_type", "exp_band"]
)["current_ctc"].transform("median")

# Calculate salary gap
filtered["gap"] = (
    filtered["current_ctc"] -
    filtered["group_median"]
)

# Top 10 underpaid professionals
underpaid = filtered.sort_values(
    by="gap"
).head(10)

if underpaid.empty:
    print("No groups contain enough members for this analysis.")
else:
    print("Top 10 Most Underpaid Professionals")
    print(
        underpaid[
            [
                "employee_id",
                "role",
                "company_type",
                "years_exp",
                "current_ctc",
                "group_median",
                "gap"
            ]
        ]
    )

Top 10 Most Underpaid Professionals
    employee_id            role company_type  years_exp  current_ctc  \
761     BLR0887  Data Scientist          MNC          2         19.7   
248     BLR0757  Data Scientist          MNC          3         20.7   
135     BLR0120  Data Scientist          MNC          3         21.8   
493     BLR0388  Data Scientist          MNC          2         26.6   
85      BLR0751  Data Scientist          MNC          2    1939999.0   
52      BLR0504  Data Scientist          MNC          2    2020000.0   
235     BLR0471  Data Scientist          MNC          3    2680000.0   
203     BLR0907  Data Scientist          MNC          2          NaN   
592     BLR0921  Data Scientist          MNC          2          NaN   

     group_median        gap  
761          26.6       -6.9  
248          26.6       -5.9  
135          26.6       -4.8  
493          26.6        0.0  
85           26.6  1939972.4  
52           26.6  2019973.4  
235          26.6  2679973

/tmp/ipykernel_1357/1206726467.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_size = data.groupby(
/tmp/ipykernel_1357/1206726467.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered["group_median"] = filtered.groupby(


#Insight:

The table identifies employees whose current CTC is lower than the median salary of others with the same role, company type, and experience band. A negative gap indicates that an employee is paid below the group median, making them relatively underpaid within their peer group.

#TASK 4

In [53]:
print("="*70)
print("               SALARY DECODER REPORT")
print("="*70)

print("\nTASK 2 - DATA CLEANING")
print("-"*70)
print("Columns converted to snake_case")
print("CTC columns converted to numeric")
print("Role names standardized")
print("Company types standardized")
print("Education tiers standardized")
print("Duplicate records removed")

print("\nTASK 3 - BUSINESS ANALYSIS")
print("-"*70)

print("Q3.1 CTC Distribution by Role")
print("Highest Paying Role :", highest_role)
print("Lowest Paying Role  :", lowest_role)

print("\nQ3.2 Experience Curve")
print(median_ctc)

print("\nQ3.3 Skill Premium")
print(result_df)

print("\nQ3.4 Company Type Premium")
print(company_ctc)
print("Unicorn Premium over MNC:", round(premium,2), "%")

print("\nQ3.5 Underpaid Professionals")
print(underpaid)

print("="*70)
print("END OF REPORT")
print("="*70)

               SALARY DECODER REPORT

TASK 2 - DATA CLEANING
----------------------------------------------------------------------
Columns converted to snake_case
CTC columns converted to numeric
Role names standardized
Company types standardized
Education tiers standardized
Duplicate records removed

TASK 3 - BUSINESS ANALYSIS
----------------------------------------------------------------------
Q3.1 CTC Distribution by Role
Highest Paying Role : PM
Lowest Paying Role  : Backend Developer

Q3.2 Experience Curve
exp_band
0-1         16.20
2-3         23.30
4-5         27.25
6+     3360000.00
Name: current_ctc, dtype: float64

Q3.3 Skill Premium
           Skill  Median CTC (With Skill)  Median CTC (Without Skill)  Premium
1             ML                    31.80                       26.05     5.75
0            AWS                    27.15                       26.20     0.95
2  System Design                    21.40                       27.30    -5.90
3     Kubernetes             

## Insights

1. PM has the highest median CTC among all roles in this dataset. Students aiming for higher salaries should study the skills and career path required for Product Management.

2. ML provides the highest median salary premium among the selected SDE skills. Learning Machine Learning can improve earning potential for software engineers.

3. Unicorn companies offer a higher median CTC than MNCs for the selected role. Candidates should compare both salary and learning opportunities before choosing between company types.


# Task 6: Code Review

## Checklist

- ✓ Notebook runs from top to bottom without errors.
- ✓ Variable names are meaningful and readable.
- ✓ Column names are converted to snake_case.
- ✓ CTC columns are converted to numeric values.
- ✓ Role names are standardized.
- ✓ Company types are standardized.
- ✓ Education tiers are standardized.
- ✓ Duplicate records are removed.
- ✓ Business questions (Q3.1 to Q3.5) are completed.
- ✓ Insights are added after each question.
- ✓ Final report is generated.
- ✓ Code is properly commented.